# disease-prediction-ml — GPU training (PhysioNet 2019)

Runs on Kaggle with **GPU + Internet** enabled. Clones the repo, installs it, prepares the attached dataset, trains, evaluates, and writes results into `artifacts/` (saved as the kernel Output).

In [ ]:
!nvidia-smi -L || echo 'no GPU'

In [ ]:
REPO = 'disease-prediction-ml'
import os, sys, subprocess
if not os.path.isdir(f'/kaggle/working/{REPO}'):
    subprocess.run(['git', 'clone', '--depth', '1',
        'https://github.com/sara-tavakoli/' + REPO + '.git'],
        cwd='/kaggle/working', check=True)
os.chdir(f'/kaggle/working/{REPO}')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)
SRC = os.path.abspath('src')
os.environ['PYTHONPATH'] = SRC + os.pathsep + os.environ.get('PYTHONPATH', '')
sys.path.insert(0, SRC)
import sepsis; print('sepsis', getattr(sepsis, '__version__', 'ok'))


## 1 · Prepare dataset

In [ ]:
# --- locate or download PhysioNet CinC-2019 psv, stage a clean root ---
import subprocess, pathlib
INP = pathlib.Path("/kaggle/input")
print("input:", [p.name for p in INP.iterdir()] if INP.exists() else "none")
setA = next((p for p in INP.rglob("training_setA") if any(p.rglob("*.psv"))), None) if INP.exists() else None
if setA is None:
    print("sepsis data not mounted -> downloading via kaggle CLI")
    dl = pathlib.Path("/kaggle/tmp/sep"); dl.mkdir(parents=True, exist_ok=True)
    subprocess.run(["kaggle","datasets","download","-d","salikhussaini49/prediction-of-sepsis",
                    "-p",str(dl),"--unzip"], check=True)
    setA = next(p for p in dl.rglob("training_setA") if any(p.rglob("*.psv")))
src_root = setA.parent
STAGE = pathlib.Path("data/physionet"); STAGE.mkdir(parents=True, exist_ok=True)
for s in ("training_setA","training_setB"):
    d = src_root/s; inner = d/"training"
    tgt = inner if inner.is_dir() and any(inner.glob("*.psv")) else d
    link = STAGE/s
    if link.exists() or link.is_symlink(): link.unlink()
    link.symlink_to(tgt.resolve())
    print(s, "->", tgt, "|", len(list(tgt.glob('*.psv'))), "stays")
ROOT = STAGE.resolve(); print("staged root:", ROOT)

## 2 · Train

In [ ]:
import subprocess
for m in ['lightgbm', 'lstm', 'gru', 'tcn', 'transformer']:
    print('=' * 20, m, '=' * 20)
    subprocess.run(['sepsis', 'train', '--config', 'configs/base.yaml',
        f'configs/model_{m}.yaml', '--set', 'data.source=physionet',
        f'data.root={ROOT}', 'data.group_by_hospital=true',
        'train.epochs=25', 'train.seed=20190804'], check=True)

## 3 · Evaluate

In [ ]:
!python scripts/update_results.py

## 4 · Show results

In [ ]:
import pathlib, IPython.display as D
for md in sorted(pathlib.Path('.').rglob('RESULTS.md')):
    D.display(D.Markdown(md.read_text()))
for png in sorted(pathlib.Path('artifacts').rglob('*.png'))[:16]:
    print(png); D.display(D.Image(str(png)))